<a href="https://colab.research.google.com/github/Shineii86/PullShark/blob/main/Pull_Shark_Automator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🦈 GitHub Pull Shark Automator

**Automate pull request creation and merging to unlock the Pull Shark achievement — all from your browser.**

---

## 🔧 How to Use

1. **Run the setup cell** (installs required library).
2. **Edit the configuration** in the second cell with your GitHub credentials.
3. **Run the main script** to start the automation.

> ⚠️ **Important**: You need a [GitHub Personal Access Token](https://github.com/settings/tokens) with `repo` scope.

---

### 🦈 PR Automation By [Shinei Nouzen](https://github.com/Shineii86)

---

In [ ]:
#@title 📦 1. Install Dependencies & Setup
!pip install -q PyGithub

import time
import random
import string
from github import Github, GithubException

print("✅ All dependencies installed and imported.")

In [ ]:
#@title ⚙️ 2. Configuration (Edit These Values)

# Replace with your actual GitHub username
GITHUB_USERNAME = "Shineii86"  #@param {type:"string"}

# Replace with your Personal Access Token (keep secret!)
GITHUB_TOKEN = "ghp_xxxxxxxxxxxxxxxxxxxx"  #@param {type:"string"}

# Repository name (must exist under your account)
REPO_NAME = "PullShark"  #@param {type:"string"}

# Number of pull requests to create and merge (minimum 2 for achievement)
NUM_PRS = 4  #@param {type:"integer"}

print(f"Configuration loaded for user '{GITHUB_USERNAME}' on repo '{REPO_NAME}'")
print(f"Will create {NUM_PRS} pull request(s).")

In [ ]:
#@title 🚀 3. Run Pull Shark Automation

# Helper functions
def generate_random_string(length=8):
    """Generate a random string for branch names and commit messages."""
    return ''.join(random.choices(string.ascii_lowercase, k=length))

def wait_for_mergeability(pr, max_wait=30):
    """Wait until GitHub confirms the PR is mergeable (or timeout)."""
    waited = 0
    while waited < max_wait:
        pr.update()
        if pr.mergeable is not False:  # True or None (still calculating)
            return True
        time.sleep(3)
        waited += 3
    return False

# Connect to GitHub
g = Github(GITHUB_TOKEN)
repo = g.get_user().get_repo(REPO_NAME)

successful_merges = 0

for i in range(NUM_PRS):
    print(f"\n--- 📦 Creating PR #{i+1} of {NUM_PRS} ---")

    # 🔁 Always get the latest commit SHA from main before branching
    main_branch = repo.get_branch("main")
    latest_commit_sha = main_branch.commit.sha
    print(f"Latest main commit: {latest_commit_sha[:7]}")

    # 1️⃣ Create a new branch from the latest main commit
    branch_name = f"auto-pr-{generate_random_string(6)}-{int(time.time())}"
    repo.create_git_ref(f"refs/heads/{branch_name}", latest_commit_sha)
    print(f"✅ Created branch: {branch_name}")

    # 2️⃣ Make a small change (update or create README.md)
    try:
        contents = repo.get_contents("README.md", ref=branch_name)
        new_content = f"{contents.decoded_content.decode()}\n- Auto-update: {generate_random_string()}"
        repo.update_file(
            contents.path,
            f"Automated update in {branch_name}",
            new_content,
            contents.sha,
            branch=branch_name
        )
        print("📝 Updated README.md")
    except GithubException:
        # README.md doesn't exist yet — create it
        repo.create_file(
            "README.md",
            f"Create README in {branch_name}",
            f"# {REPO_NAME}\n\nAuto-generated: {generate_random_string()}\n\n---\n🦈 PR Automation By [Shinei Nouzen](https://github.com/Shineii86)",
            branch=branch_name
        )
        print("📄 Created README.md")

    # 3️⃣ Create the Pull Request
    pr = repo.create_pull(
        title=f"Auto-PR {generate_random_string(4)} #{i+1}",
        body="🤖 Automated pull request for the Pull Shark achievement.\n\n---\n🦈 PR Automation By [Shinei Nouzen](https://github.com/Shineii86)",
        head=branch_name,
        base="main"
    )
    print(f"🔗 Created PR: {pr.html_url}")

    # 4️⃣ Wait for mergeability check to complete
    print("⏳ Waiting for GitHub to calculate mergeability...")
    if not wait_for_mergeability(pr):
        print("❌ PR not mergeable after waiting. Stopping.")
        break

    # 5️⃣ Merge the PR
    try:
        pr.merge(merge_method="merge")
        print(f"🎉 Merged PR #{pr.number}")
        successful_merges += 1
    except GithubException as e:
        print(f"❌ Merge failed: {e}")
        break

    # 6️⃣ Wait for GitHub to fully update the main branch
    print("⏸️  Pausing 10 seconds for GitHub to process the merge...")
    time.sleep(10)

print(f"\n🏁 Finished. Successfully merged {successful_merges} out of {NUM_PRS} pull requests.")
if successful_merges >= 2:
    print("🦈 Congratulations! You've met the requirements for Pull Shark!")
else:
    print("⚠️ You need at least 2 merged PRs for the achievement.")

---

### 📚 Need Help?

- **Pull Shark not appearing?** Wait a few minutes and refresh your GitHub profile.
- **"405 Not mergeable" error?** Go to repo **Settings → General → Pull Requests** and enable **"Allow auto-merge"**.
- **Bad credentials?** Double‑check your Personal Access Token and ensure it has `repo` scope.

---

<div align="center">

🦈 **PR Automation By [Shinei Nouzen](https://github.com/Shineii86)**  
*Made with ❤️ for GitHub achievement hunters*

</div>